In [0]:
# importar bibliotecas
import re
import unicodedata
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.functions import col

In [0]:
#👉 Função para padronizar nomes de colunas

def clean_column_name(col_name):
    col_name = unicodedata.normalize('NFKD', col_name)
    col_name = col_name.encode('ascii', 'ignore').decode('utf-8')
    col_name = col_name.lower()
    col_name = col_name.replace(" ", "_")
    col_name = re.sub(r'[^a-z0-9_]', '', col_name)
    col_name = re.sub(r'_+$', '', col_name)   # ← remove _ somente no final
    return col_name

In [0]:
# Elimina duplicados após a limpeza

def make_unique_columns(columns):
    counts = {}
    new_cols = []

    for c in columns:
        clean_c = clean_column_name(c)
        if clean_c in counts:
            counts[clean_c] += 1
            clean_c = f"{clean_c}_{counts[clean_c]}"
        else:
            counts[clean_c] = 0
        new_cols.append(clean_c)

    return new_cols

In [0]:
# Definir caminhos

source_path = "/Volumes/workspace/pipeline_estudo/raw_files/censo_padre_bernardo/"
checkpoint_path = "/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/covid_19/bronze" 
# salva o "ponteiro" de onde o Spark parou. Se você rodar o código hoje e amanhã, ele só processará os arquivos que surgiram nesse intervalo.

In [0]:
# Lendo os dados 

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(source_path)
)

In [0]:
# Lê os arquivos com Autoloader

schema = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
    .schema
)

df = (
    spark.readStream
    .format("cloudFiles") # Isso diz ao Spark para usar o mecanismo Auto Loader
    .option("cloudFiles.format", "csv") # Isso diz que os arquivos que ele vai vigiar são CSV
    .option("pathGlobFilter", "covid_19_caso_full.csv.gz") # lendo o arquvo covid_19_caso_full.csv.gz
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .schema(schema)
    .load(source_path)
)

df_filtrado = df.filter(
    (col("state") == "GO") & 
    (col("city") == "Padre Bernardo")
)


In [0]:
# Aplica padronização  das colunas
df_bronze = df.toDF(*make_unique_columns(df.columns))

In [0]:
# Adiciona colunas técnicas da Bronze

df_bronze = (
    df_filtrado
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("sales.csv"))
)

In [0]:
# Cria a pasta de checkpoint aqui sempre deve ser criar uma pasta para cada assunto
dbutils.fs.mkdirs("/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/covid_19/bronze")

In [0]:
from pyspark.sql.functions import col

if "is_last" in df_bronze.columns:
    df_bronze = df_bronze.withColumn("is_last", col("is_last").cast("string"))

In [0]:
# Grava o stream na tabela Bronze final
(df_bronze.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_path)
 .option("mergeSchema", "true") # Isso permite que o Delta ajuste a tabela automaticamente se encontrar colunas novas ou mudanças compatíveis
 .trigger(availableNow=True)
 .toTable("workspace.drs_bronze.covid_19"))

In [0]:
# Valida a tabela Bronze final
spark.sql("""
SELECT *
FROM workspace.drs_bronze.covid_19
LIMIT 10
""").display()

In [0]:
# Conta os registros da Bronze
spark.sql("""
SELECT COUNT(*) AS total_bronze_v2
FROM workspace.drs_bronze.sales_v2
""").display()

In [0]:
# Valida duplicidade por chave de negócio

spark.sql("""
SELECT
    city_ibge_code,
    last_available_date,
    COUNT(*) AS qtd
FROM workspace.drs_bronze.covid_19
GROUP BY city_ibge_code, last_available_date
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()